<a href="https://colab.research.google.com/github/Jun-1112/FYP-project-Trunk-and-weed-detection-for-agricultural-usage/blob/main/Object_Detection_(Weed).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup
!pip install ultralytics -q

from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
# To extract the external weed dataset from Drive
images_zip = '/content/drive/MyDrive/Colab Notebooks/intel Real Sense Depth_Clicks.zip'
labels_zip = '/content/drive/MyDrive/Colab Notebooks/weed_dataset.zip'

print("Unzipping images (large archive, may take 1-2 minutes)...")
!unzip -q "{images_zip}" -d /content/raw_images

print("Unzipping annotations...")
!unzip -q "{labels_zip}" -d /content/raw_labels

print("Extraction complete.")
print("raw_images contents:", os.listdir('/content/raw_images')[:5])
print("raw_labels contents:", os.listdir('/content/raw_labels')[:5])

In [ ]:
# To verify which class IDs are actually present in the labels
label_root = '/content/raw_labels/weed_dataset/YOLO_darknet'

class_ids = set()
for root, dirs, files in os.walk(label_root):
    for file in files:
        if file.endswith('.txt'):
            with open(os.path.join(root, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        class_ids.add(int(parts[0]))

print("Class IDs found in annotations:", sorted(class_ids))
print("Number of distinct classes:", len(class_ids))

In [ ]:
# To match image-label pairs and build 80/20 train/val split
import shutil, random

image_root = '/content/raw_images'
output_root = '/content/weed_yolo_dataset'

for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(output_root, folder), exist_ok=True)

# Collect all images (recursively, since the archive is nested)
image_files = []
for root, dirs, files in os.walk(image_root):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, file))

# Index available labels by filename stem
label_names = {
    os.path.splitext(file)[0]
    for file in os.listdir(label_root)
    if file.endswith('.txt')
}

# Keep only images that have a matching annotation file
pairs = []
for image_path in image_files:
    stem = os.path.splitext(os.path.basename(image_path))[0]
    if stem in label_names:
        pairs.append((image_path, os.path.join(label_root, stem + '.txt')))

print("Total images found:", len(image_files))
print("Total label files:", len(label_names))
print("Matched image-label pairs:", len(pairs))

random.seed(42)
random.shuffle(pairs)

split_index = int(len(pairs) * 0.8)
train_pairs = pairs[:split_index]
val_pairs = pairs[split_index:]

def copy_pairs(pairs, split):
    for image_path, label_path in pairs:
        shutil.copy2(image_path, os.path.join(output_root, 'images', split, os.path.basename(image_path)))
        shutil.copy2(label_path, os.path.join(output_root, 'labels', split, os.path.basename(label_path)))

copy_pairs(train_pairs, 'train')
copy_pairs(val_pairs, 'val')

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

In [ ]:
# To write data.yaml
import yaml

class_names = [
    'Commelina benghalensis',
    'Cyperus rotundus',
    'Chenopodium album (Lambsquarter)',
    'Malva parviflora (Little mallow)',
    'Euphorbia geniculata',
    'Ipomoea obscura (Obscure morning glory)',
    'Clitoria ternatea (Asian pigeonwings)',
    'Argemone mexicana',
    'Euphorbia hirta',
    'Digitaria sp.',
    'Parthenium hysterophorus (Congress grass)',
    'Euphorbia hypericifolia (Graceful sandmat)',
    'Senna obtusifolia (Sicklepod)',
    'Cynodon dactylon (Bermuda grass)',
    'Cassia sp. (Dwarf cassia)'
]

data_yaml = {
    'path': output_root,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 15,
    'names': class_names
}

with open(f'{output_root}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(open(f'{output_root}/data.yaml').read())

In [ ]:
# To check split counts
for split in ['train', 'val']:
    images = os.listdir(f'{output_root}/images/{split}')
    labels = os.listdir(f'{output_root}/labels/{split}')
    print(f"{split}:  images={len(images)}  labels={len(labels)}")

In [ ]:
# To train the weed detection model (YOLOv8n)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{output_root}/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=25,
    device=0,
    plots=True,
    project='/content/runs/detect',
    name='weed_detection_model',
    save_period=10,
)

print("Training complete. Best weights saved.")

In [ ]:
# To back up weights, plots, metrics, and config to Drive
backup_dir = '/content/drive/MyDrive/Colab Notebooks/detfyp_backup/weed_results'
os.makedirs(backup_dir, exist_ok=True)

run_dir = '/content/runs/detect/weed_detection_model'

# Weights
shutil.copy(f'{run_dir}/weights/best.pt', f'{backup_dir}/best_weeddet.pt')
shutil.copy(f'{run_dir}/weights/last.pt', f'{backup_dir}/last_weeddet.pt')

# Config + metrics
shutil.copy(f'{output_root}/data.yaml', f'{backup_dir}/data.yaml')
shutil.copy(f'{run_dir}/results.csv', f'{backup_dir}/results.csv')
shutil.copy(f'{run_dir}/args.yaml', f'{backup_dir}/args.yaml')

# All generated plots and batch visualisations
for f in os.listdir(run_dir):
    if f.endswith('.png') or f.endswith('.jpg'):
        shutil.copy(f'{run_dir}/{f}', f'{backup_dir}/{f}')

print("Weights, data.yaml, results.csv, args.yaml, and all plots backed up to Drive.")
